# 🧠 Gaussian Naive Bayes Explanation and Hands-on Example

Welcome to the hands-on explanation notebook for **Naive Bayes**! In this notebook, we will:
1. Generate a synthetic dataset based on the bounding box validation case study (`real-object` vs. `false-positive`).
2. Train a **Gaussian Naive Bayes** classifier using `scikit-learn`.
3. Visualize the quadratic decision boundaries learned by the model.
4. Implement **Gaussian Naive Bayes from scratch** using pure Python/NumPy:
   - Learn class priors, feature means ($\mu$), and variances ($\sigma^2$) for each class.
   - Calculate Gaussian probability densities.
   - Compute log-posteriors to avoid numerical underflow:
     $$\log P(C_k | \mathbf{x}) \propto \log P(C_k) - \frac{1}{2} \sum_{j=1}^{n} \left[ \log(2\pi \sigma_{k,j}^2) + \frac{(x_j - \mu_{k,j})^2}{\sigma_{k,j}^2} \right]$$
5. Evaluate our scratch implementation and compare its predictions with scikit-learn.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Case Study Data Generation

We generate 100 sample bounding boxes with two continuous features:
1.  `ratio` (height-to-width ratio)
2.  `center_y` (vertical center pixel coordinate)

Classes:
*   Class 1 (`real-object`): centered at ratio 1.5, center_y 240 (middle of the screen).
*   Class 0 (`false-positive`): centered at ratio 0.8, center_y 420 (ground level shadows creating fake bounding boxes).

In [ ]:
m = 100

# Class 1: Real Objects
X_real = np.random.randn(m // 2, 2) * np.array([0.25, 40]) + np.array([1.5, 240])
y_real = np.ones(m // 2)

# Class 0: False Positives (Shadows/Noise)
X_fake = np.random.randn(m // 2, 2) * np.array([0.20, 30]) + np.array([0.8, 420])
y_fake = np.zeros(m // 2)

# Combine datasets
X_train = np.vstack((X_real, X_fake))
y_train = np.concatenate((y_real, y_fake))

# Plot the dataset
plt.figure(figsize=(8, 5))
plt.scatter(X_real[:, 0], X_real[:, 1], color='blue', label='Class 1: Real Object', alpha=0.7)
plt.scatter(X_fake[:, 0], X_fake[:, 1], color='red', label='Class 0: False Positive', alpha=0.7)
plt.xlabel('Height-to-Width Ratio')
plt.ylabel('Center Y Coordinate')
plt.title('Bounding Box Verification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Gaussian Naive Bayes using Scikit-Learn

Let's fit the scikit-learn `GaussianNB` model and evaluate the learned class priors, means, and variances.

In [ ]:
# Train model
gnb_sklearn = GaussianNB()
gnb_sklearn.fit(X_train, y_train)

# Print parameters
print("--- Class Parameters Learned by Scikit-Learn ---")
print("Class Priors    :", gnb_sklearn.class_prior_)
print("Class Means (μ) :\n", gnb_sklearn.theta_)
print("Class Variances (σ²):\n", gnb_sklearn.var_)

# Evaluate accuracy
y_pred_sklearn = gnb_sklearn.predict(X_train)
print(f"\nTraining Accuracy: {accuracy_score(y_train, y_pred_sklearn) * 100:.2f}%")

## 3. Gaussian Naive Bayes from Scratch

Let's implement the model using NumPy.
During `fit`:
1.  Calculate class priors $P(C_k) = \frac{m_k}{m}$.
2.  Compute the mean $\mu_{k,j}$ and variance $\sigma_{k,j}^2$ for each feature $j$ and class $k$.

During `predict`:
Compute the log posterior probability for each class $C_k$:
$$\log P(C_k | \mathbf{x}) \propto \log P(C_k) + \sum_{j=1}^{n} \log P(x_j | C_k)$$

Where:
$$\log P(x_j | C_k) = -\frac{1}{2} \log(2\pi \sigma_{k,j}^2) - \frac{(x_j - \mu_{k,j})^2}{2\sigma_{k,j}^2}$$

And predict the class with the maximum log posterior.

In [ ]:
class CustomGaussianNB:
    def __init__(self):
        self.classes = None
        self.priors = {}
        self.means = {}
        self.vars = {}

    def fit(self, X, y):
        self.classes = np.unique(y)
        m = X.shape[0]
        
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / m
            self.means[c] = np.mean(X_c, axis=0)
            self.vars[c] = np.var(X_c, axis=0)

    def _pdf(self, x, mean, var):
        # Gaussian probability density function
        num = np.exp(-((x - mean) ** 2) / (2 * var))
        den = np.sqrt(2 * np.pi * var)
        return num / den

    def _predict_single(self, x):
        posteriors = []
        
        for c in self.classes:
            log_prior = np.log(self.priors[c])
            eps = 1e-15
            pdfs = self._pdf(x, self.means[c], self.vars[c])
            log_likelihood = np.sum(np.log(pdfs + eps))
            
            log_posterior = log_prior + log_likelihood
            posteriors.append((log_posterior, c))
            
        return max(posteriors)[1]

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

# Train custom scratch model
gnb_scratch = CustomGaussianNB()
gnb_scratch.fit(X_train, y_train)

# Evaluate Custom GNB
y_pred_scratch = gnb_scratch.predict(X_train)
scratch_acc = accuracy_score(y_train, y_pred_scratch)

print(f"Custom Scratch Gaussian NB Accuracy: {scratch_acc * 100:.2f}%")

## 4. Visualizing the Decision Boundary

Because Gaussian Naive Bayes models the variances of each class independently, it can draw curved, quadratic decision boundaries. Let's visualize this!

In [ ]:
from matplotlib.colors import ListedColormap

# Create grid for boundaries
x_min, x_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
y_min, y_max = X_train[:, 1].min() - 30, X_train[:, 1].max() + 30
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01), 
                     np.arange(y_min, y_max, 1))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Predict on grid
Z = gnb_scratch.predict(grid_points)
Z = Z.reshape(xx.shape)

# Plot decision boundary partitions
plt.figure(figsize=(10, 6))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['red', 'blue']

plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.5)
plt.scatter(X_real[:, 0], X_real[:, 1], color='blue', label='Class 1: Real Object', alpha=0.6, edgecolor='k')
plt.scatter(X_fake[:, 0], X_fake[:, 1], color='red', label='Class 0: False Positive', alpha=0.6, edgecolor='k')

plt.xlabel('Height-to-Width Ratio')
plt.ylabel('Center Y Coordinate')
plt.title('Gaussian Naive Bayes Decision Boundary')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

Notice the non-linear, curved parabolic decision boundary, which is perfectly suited to separate concentric or shifted normal distributions of continuous data points!

## 💡 Connection to Deep Learning & YOLO
*   **Prior Probability Initialization:** In object detection models (like YOLO), the class classification layers are initialized with specific **priors**. Since background clutter is much more common than actual objects at the start of training, the bias weights of the classification heads are initialized using prior ratios (e.g. $b = -\log((1 - p)/p)$ where $p$ is the object prior probability, e.g., $0.01$). This matches Bayes' Theorem priors and prevents the model from exploding its losses in early training epochs.